# Phase 7 — Analysis & Write-up (Colab)

**Pre-requisite:** Phases 3-6 done — all four result JSONs in Drive:
`robustness_baseline.json`, `robustness_robust.json`, `mc_dropout_metrics.json`,
`conformal_metrics.json`.

Packages the project into a portfolio artifact: final results table, 3 key figures
at 300 dpi, generated README, resume bullet, and a project-completion audit.

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = '/content/drive/MyDrive/robot-perception'
RESULTS     = f'{PROJECT_DIR}/results'
RESULTS_DIR = f'{RESULTS}/figures'

CLASS_NAMES = ['arm', 'leg', 'torso', 'head', 'sensor']
CORRUPTIONS = ['gaussian_noise', 'motion_blur', 'gaussian_blur', 'brightness', 'occlusion']
SEVERITIES  = [1, 2, 3, 4, 5]

def load(name):
    with open(f'{RESULTS}/{name}') as f:
        return json.load(f)

baseline  = load('robustness_baseline.json')
robust    = load('robustness_robust.json')
mc        = load('mc_dropout_metrics.json')
conformal = load('conformal_metrics.json')
print('All result JSONs loaded.')

## Step 1 — Final results table

In [ ]:
table = pd.DataFrame([
    {'Model': 'RT-DETR baseline',
     'Clean mAP': round(baseline['clean_mAP50'], 3),
     'mPC': round(baseline['mPC'], 3),
     'Relative mPC': round(baseline['relative_mPC'], 3),
     'ECE': '-'},
    {'Model': 'RT-DETR + robust aug',
     'Clean mAP': round(robust['clean_mAP50'], 3),
     'mPC': round(robust['mPC'], 3),
     'Relative mPC': round(robust['relative_mPC'], 3),
     'ECE': '-'},
    {'Model': 'RT-DETR + MC Dropout',
     'Clean mAP': round(robust['clean_mAP50'], 3),
     'mPC': round(robust['mPC'], 3),
     'Relative mPC': round(robust['relative_mPC'], 3),
     'ECE': round(mc['ece'], 3)},
])
print(table.to_string(index=False))
table.to_csv(f'{RESULTS}/final_results_table.csv', index=False)

fig, ax = plt.subplots(figsize=(9, 2.2))
ax.axis('off')
t = ax.table(cellText=table.values, colLabels=table.columns,
             cellLoc='center', loc='center')
t.auto_set_font_size(False); t.set_fontsize(11); t.scale(1, 1.6)
plt.title('Final Results', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/final_results_table.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 2 — Three key figures at 300 dpi

In [ ]:
# KEY FIG 1 — robustness curve, baseline vs robust
fig, axes = plt.subplots(1, len(CORRUPTIONS), figsize=(20, 4))
for i, corruption in enumerate(CORRUPTIONS):
    base = [baseline['results_grid'][corruption][str(s)] for s in SEVERITIES]
    rob  = [robust['results_grid'][corruption][str(s)] for s in SEVERITIES]
    axes[i].plot(SEVERITIES, base, 'r-o', label='baseline')
    axes[i].plot(SEVERITIES, rob,  'g-o', label='robust')
    axes[i].axhline(y=baseline['clean_mAP50'], color='k', linestyle='--')
    axes[i].set_title(corruption); axes[i].set_xlabel('Severity')
    axes[i].set_xticks(SEVERITIES); axes[i].legend()
axes[0].set_ylabel('mAP@0.5')
plt.suptitle('Key Figure 1 — Robustness curve: baseline (red) vs robust (green)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/key_fig1_robustness.png', dpi=300)
plt.show()

In [ ]:
# KEY FIG 2 — reliability diagram (re-use VIZ 5.A saved image at 300 dpi)
# The reliability diagram is data-derived in Phase 5; here we re-render the
# saved figure as the high-res key figure.
import shutil
src = f'{RESULTS_DIR}/viz5a_reliability_diagram.png'
dst = f'{RESULTS_DIR}/key_fig2_reliability.png'
if os.path.exists(src):
    shutil.copy2(src, dst)
    from IPython.display import Image, display
    display(Image(dst))
    print('Key Figure 2 saved (reliability diagram).')
else:
    print('viz5a_reliability_diagram.png not found — re-run Phase 5 VIZ 5.A.')

In [ ]:
# KEY FIG 3 — uncertainty vs severity (from mc_dropout_metrics.json)
unc_by_corruption = mc['uncertainty_by_corruption']
plt.figure(figsize=(9, 5))
for corruption, means in unc_by_corruption.items():
    plt.plot(SEVERITIES, means, marker='o', label=corruption)
plt.xlabel('Corruption Severity'); plt.ylabel('Mean Uncertainty')
plt.title('Key Figure 3 — Uncertainty increases with corruption severity')
plt.xticks(SEVERITIES); plt.legend()
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/key_fig3_uncertainty.png', dpi=300)
plt.show()

## Step 3 — Generate README content

In [ ]:
improvement = (robust['mPC'] - baseline['mPC']) / baseline['mPC'] * 100

readme = f'''# Robust + Uncertainty-Aware Robot Perception

An uncertainty-aware RT-DETR detector for humanoid robot components that degrades
gracefully under real-world image corruptions and outputs calibrated confidence.

## Results

| Model | Clean mAP | mPC | Relative mPC | ECE |
|---|---|---|---|---|
| RT-DETR baseline | {baseline["clean_mAP50"]:.3f} | {baseline["mPC"]:.3f} | {baseline["relative_mPC"]:.3f} | - |
| RT-DETR + robust aug | {robust["clean_mAP50"]:.3f} | {robust["mPC"]:.3f} | {robust["relative_mPC"]:.3f} | - |
| RT-DETR + MC Dropout | {robust["clean_mAP50"]:.3f} | {robust["mPC"]:.3f} | {robust["relative_mPC"]:.3f} | {mc["ece"]:.3f} |

## Key finding

Corruption-aware augmentation improved mean performance under corruption by
{improvement:.1f}% (mPC: {baseline["mPC"]:.3f} -> {robust["mPC"]:.3f}) without
sacrificing clean accuracy. MC Dropout provides calibrated uncertainty
(ECE: {mc["ece"]:.3f}) that increases monotonically with corruption severity.
Conformal prediction at epsilon=0.05 achieves {conformal["coverage_clean"]*100:.1f}%
empirical coverage on the clean test set.

## Methods

- Architecture: RT-DETR-L fine-tuned from COCO pretrained
- Corruption benchmark: ImageNet-C methodology, 5 corruption types x 5 severities
- Uncertainty: MC Dropout, N={mc["n_passes"]} stochastic forward passes
- Conformal prediction: epsilon=0.05, empirical coverage {conformal["coverage_clean"]:.3f}
'''
print(readme)

# Optionally write to Drive
with open(f'{PROJECT_DIR}/README_generated.md', 'w') as f:
    f.write(readme)
print('\nWritten: README_generated.md (copy into the repo README.md results section)')

## Step 4 — Resume bullet

In [ ]:
bullet = f'''Robust Humanoid Robot Perception - RT-DETR . MC Dropout . Conformal Prediction . PyTorch
- Built an uncertainty-aware RT-DETR detector for humanoid robot components;
  benchmarked robustness across 25 corruption scenarios (5 types x 5 severities,
  ImageNet-C methodology), achieving mAP@0.5: {baseline["clean_mAP50"]:.2f} on clean images
- Designed a corruption-aware augmentation pipeline that improved mean performance
  under corruption (mPC) by {improvement:.0f}% relative to baseline without sacrificing clean accuracy
- Implemented MC Dropout uncertainty quantification (N={mc["n_passes"]} stochastic passes);
  achieved ECE {mc["ece"]:.2f} with uncertainty increasing monotonically with corruption
  severity; applied conformal prediction (eps=0.05) for {conformal["coverage_clean"]*100:.0f}% coverage guarantees'''
print(bullet)

## Step 5 — Project-completion audit

In [ ]:
checks = [
    ('Phase 2: clean mAP@0.5 >= 0.70', baseline['clean_mAP50'] >= 0.70,
     f"{baseline['clean_mAP50']:.3f}"),
    ('Phase 3: all 25 corrupted sets evaluated',
     all(str(s) in baseline['results_grid'][c]
         for c in CORRUPTIONS for s in SEVERITIES), '25/25'),
    ('Phase 4: mPC improvement >= 10%',
     improvement >= 10.0, f'{improvement:.1f}%'),
    ('Phase 4: clean mAP drop < 0.03',
     baseline['clean_mAP50'] - robust['clean_mAP50'] < 0.03,
     f"{baseline['clean_mAP50'] - robust['clean_mAP50']:.3f}"),
    ('Phase 5: ECE <= 0.10', mc['ece'] <= 0.10, f"{mc['ece']:.3f}"),
    ('Phase 5: Spearman rho > 0.3', mc['spearman_rho'] > 0.3,
     f"{mc['spearman_rho']:.3f}"),
    ('Phase 5: >= 3/5 corruptions hit 1.5x severity ratio',
     sum(r >= 1.5 for r in mc['severity_ratios'].values()) >= 3,
     f"{sum(r >= 1.5 for r in mc['severity_ratios'].values())}/5"),
    ('Phase 6: coverage >= 0.95', conformal['coverage_clean'] >= 0.95,
     f"{conformal['coverage_clean']:.3f}"),
    ('Phase 6: avg set size clean <= 1.5',
     conformal['avg_set_size_clean'] <= 1.5,
     f"{conformal['avg_set_size_clean']:.3f}"),
    ('Phase 6: avg set size severity 5 > 2.0',
     conformal['set_size_by_severity']['5'] > 2.0,
     f"{conformal['set_size_by_severity']['5']:.3f}"),
]

print(f'{"CHECK":<52}{"VALUE":<12}STATUS')
print('-' * 72)
all_pass = True
for name, ok, val in checks:
    all_pass &= ok
    print(f'{name:<52}{val:<12}{"PASS" if ok else "FAIL"}')
print('-' * 72)
print('PROJECT COMPLETE' if all_pass else 'INCOMPLETE — address FAIL rows above')

## Phase 7 Completion Checklist

- [ ] Final results table complete with real numbers (CSV + PNG saved)
- [ ] 3 key figures saved at 300 dpi
- [ ] README content generated and merged into repo `README.md`
- [ ] Resume bullet drafted with actual numbers
- [ ] Project-completion audit: all rows PASS
- [ ] GitHub repo public, clean, working `requirements.txt`